В этом ноутбуке мы проверим, действительно ли тулы в тестовых файлах Gi_tool и Gi_cat zero shot

In [27]:
import json
from tqdm import tqdm
import os
from toolbench.utils import (
    standardize,
    change_name,
    replace_llama_with_condense
)

In [2]:
with open('data/toolllama_G123_dfs_train.json', 'r') as f:
    data = json.load(f)

Соберем тулы, которые есть в трейне

In [18]:
train_tool_set = set()
for item in tqdm(data):
    prompt = item['conversations'][0]['value']
    descriptions = prompt.split('Specifically, you have access to the following APIs: ')[1]
    while len(descriptions.split("'name': '", 1)) > 1:
        name, descriptions = descriptions.split("'name': '", 1)[1].split("', 'description':", 1)
        train_tool_set.add(name)

100%|███████████████████████████████████████████████████████████████████| 26999/26999 [00:00<00:00, 47126.03it/s]


In [20]:
len(train_tool_set)

10389

In [32]:
list(train_tool_set)[:10]

['nftsearch_for_nft_explorer',
 'get_word_by_length_and_start_for_random_word_api',
 'by_filter_for_netflix_original_series_top_100_ranked',
 'get_countries_for_cities_cost_of_living_and_average_prices_api',
 'search_videos_channels_playlists_for_youtube_search_and_download',
 'countrieslist_for_geosource_api',
 'artist_songs_for_genius_song_lyrics',
 'getliveevents_for_sport_odds',
 'categories_for_rugbyapi2',
 'search_for_wayback_machine']

Теперь выберем каждый файл и проверим, сколько пересечений в каждом

In [55]:
dir = 'data/test_data_merged/'
for file in os.listdir(dir):
    print(file)
    curr_names = set()
    if file.endswith('json'):
        with open(dir+file, 'r') as f:
            curr_data = json.load(f)
            for item in curr_data:
                for tool in item['api_list']:
                    tool_name = standardize(tool['tool_name'])
                    api_name = change_name(standardize(tool["api_name"]))
                    name = f"{api_name}_for_{tool_name}"
                    curr_names.add(name)
        print('Num tools: ', len(curr_names))
        print('Num in train: ', len(train_tool_set.intersection(curr_names)))
        # print(curr_names.difference(train_tool_set))
        print(len(train_tool_set.intersection(curr_names)) / len(curr_names))
    print('#'*100)

G2_inst.json
Num tools:  561
Num in train:  497
0.8859180035650623
####################################################################################################
G3_inst.json
Num tools:  348
Num in train:  335
0.9626436781609196
####################################################################################################
G1_tool.json
Num tools:  390
Num in train:  298
0.764102564102564
####################################################################################################
G1_inst.json
Num tools:  419
Num in train:  405
0.9665871121718377
####################################################################################################
G2_cat.json
Num tools:  358
Num in train:  54
0.15083798882681565
####################################################################################################
G1_cat.json
Num tools:  265
Num in train:  36
0.13584905660377358
################################################################################################

In [54]:
'get_1_3_list_non_working_days_for_working_days' in train_tool_set

False